<h1 align="center">SWDB Problem Set: Becoming a Data Detective</h1>
<h3 align="center">From someone else's figure to your own analysis</h3>
<p align="center"><i>Works with any SWDB dataset &mdash; bring the one your chosen figure came from.</i></p>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>How this problem set works</h2>

This morning you explored a dataset and made figures. Those figures are now posted on Slack.

**Your starting point is one of your classmates' figures.** Pick any figure from the channel, along
with the dataset it came from &mdash; ideally one you did *not* work on this morning.

| Part | Task |
| --- | --- |
| 1 | Load their dataset and find the pieces the figure needs |
| 2 | Reproduce the figure, and interrogate what it shows |
| 3 | Align activity to event onsets: raster and PSTH |
| 4 | Signal and noise correlations, and whether to trust them |

You already have the data-access skills for Part 1 from this morning's tutorial. This problem set is
about what comes after loading: **shaping data, and checking whether the result means anything.**

**Deliverable:** a short README naming the figure and dataset you chose, the decisions you made at
each step, and an honest assessment of what your numbers do and do not support.

<b>Every dataset is different, and the notebook does not know which one you picked.</b> The code
cells are prompts, not templates &mdash; you write what goes in them, using the access patterns from
this morning. Only a few things are given: the imports, and two helper functions from the tutorial.

The differences you will run into are not cosmetic. Across the datasets in this workshop:

- **Recording modality** &mdash; a continuous calcium signal in some, discrete spike times in
  others. Spikes need binning before anything here applies.
- **Sampling rate** &mdash; from a few Hz to tens of kHz, which sets what timing you can resolve.
- **Number of neurons** &mdash; tens to thousands, which changes what is tractable in one pass.
- **Stimulus structure** &mdash; many conditions with few repeats, few conditions with many, or no
  sensory stimulus at all.
- **What was recorded alongside** &mdash; running, licking, pupil, reward; some datasets have all of
  it, some none.
- **Where things live in the file** &mdash; container and column names differ, and so does which
  container holds the trial table.

None of that is written on the outside of the file. **You have to look.** Part of each prompt is
deciding whether the analysis it asks for even applies to your dataset &mdash; and saying so when it
does not.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Taking it slow: Analysis step by step</h2>

You can now generate an analysis faster than you can check one. Ask an LLM for a correlation matrix
and you will have one in thirty seconds, beautifully formatted, with a colorbar.

The problem is that a result computed on four trials can look exactly like a result computed on four
hundred. A bug can look exactly like a finding. A correlation computed in a window where nothing
happened can look exactly like a real effect.

So the questions to keep asking are:

- **What is actually in this file?** Not what you assume &mdash; what is there.
- **Does this dataset support the question I am asking?**
- **How is the data being transformed?** Plot the data after each step.
- **What would make this result wrong?** Name it before you see the answer.

</div>

---

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pynwb
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

data_dir = '/data'

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 1: Load the dataset and find the pieces you need</h1>

Same access pattern as this morning: find your dataset's mount under <code>/data</code>, locate a
session's NWB file, then dot and bracket notation into the containers.

**Your classmate's figure tells you what to look for.** Before you open anything, list the pieces the
figure needs &mdash; neural activity, plus whatever else it plots: a behavioral trace, epoch
boundaries, trial times, stimulus identity.

Then find each one, and note the ones that turn out not to exist. **A piece being absent is a
finding about the dataset, not a failure.** Some datasets have no running wheel, no pupil camera, no
visual stimulus at all. You will build the figure from what is there.

</div>

In [ ]:
# List the datasets mounted under /data.

In [ ]:
# EDIT: read your dataset's metadata CSV from /code/metadata and look at what it
# offers: how many subjects, how many session types, how many sessions each.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Which session does your classmate's figure come from? Use the table to find it
&mdash; subject, session type, date &mdash; and say what you filtered on.

Look at what the table offers before you filter. How many subjects, how many session types, how many
sessions each? That inventory is the first thing you know about the dataset.

**Then ask what kind of neurons you are recording from.** This is not a detail &mdash; it decides
what your population average means. Check the transgenic line, the virus, and any other metadata
describing what was labeled (`nwb.subject.genotype`, the imaging plane's `indicator`, the session
metadata table).

- **Imaging.** You see only the cells expressing the calcium indicator. A pan-excitatory driver
  gives you a very different population from an interneuron-specific one, and "population activity"
  in each case means something different.
- **Electrophysiology.** A probe records whatever is near it, so the recording is not cell-type
  specific by default. But a line or virus may still be present for **optotagging** &mdash; light
  activation used to identify a targeted cell type among the recorded units. If so, there may be a
  column marking which units were tagged.

Write down what is labeled in your session, and say what population your averages are actually
averaging over.

</div>

In [ ]:
# EDIT: filter the table to the session behind your figure, take one row as
# `session`, and say what you filtered on.

In [ ]:
# EDIT: examine the column values of the session you selected
# what is the session type, genotype, the targeted structure, etc. 

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<b>Now build the path and load the data.</b> The metadata table's <code>name</code> column is normally the session's
folder name inside the mount, so you can go straight there rather than searching. Inside that folder
sits one NWB store &mdash; either a single <code>.nwb</code> file (HDF5) or a <i>directory</i>
(zarr).

</div>


In [ ]:
# EDIT: set `dataset_dir` to the mount holding your dataset (one of the names
# printed above), then join it with your session's folder name to get `session_dir`.


In [ ]:
# Open the NWB file. Name it `nwb`.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Find the data the figure needs</h3>

A handful of containers hold almost everything. Which one holds what **varies by dataset**, so list
them all before you index into any of them.

| container | commonly holds |
| --- | --- |
| `processing` | processed neural activity &mdash; in some datasets also behavior |
| `intervals` | epoch tables, trial tables, stimulus presentation tables |
| `stimulus` | stimulus templates &mdash; but in some datasets, the trial tables too |
| `acquisition` | raw acquired signals |
| `events` | discrete behavioral and stimulus events, in some datasets |

<div>


</div>


In [ ]:
# What is in this file? Print the containers before you index into any of them:
# processing, intervals, acquisition, stimulus. Then look INSIDE each processing
# module -- the listing above only gives you the module names. 


In [ ]:
# What tables are present in each container? What are the columns of those tables? 

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Is your dataset continuous or spiking?</b> This is the first fork in the road, and it
changes what "activity" even means.

<b>Continuous</b> (calcium imaging, LFP): a `(n_timepoints, n_cells)` array already exists in the
file. Find it and you are done.

<b>Spiking</b> (Neuropixels, sorted electrophysiology): there is no such array. Each unit carries its
own list of spike times, usually in a `units` table, and you must <b>bin</b> them yourself &mdash;
choose a bin width, count spikes per bin, divide by the width to get a rate in spikes/s. Everything
downstream then works the same way.

Two decisions come with spiking data, and neither has a default:

- <b>Which units.</b> Spike sorting produces more units than you should analyze. There will be
  quality-control columns (`is_qc_pass`, `firing_rate`, `presence_ratio`, `snr`) and often an
  anatomical label. Select on them explicitly and say what you selected &mdash; a session can drop
  from thousands of units to dozens, and the ones you drop change your answer.
- <b>Bin width.</b> Too wide blurs the response; too narrow leaves mostly-empty bins and noisy
  single-trial estimates. Try a few and see how much your answer moves.

<pre>
bin_width = 0.010                                    # seconds -- your decision
edges = np.arange(0, t_end + bin_width, bin_width)
counts, _ = np.histogram(one_unit_spike_times, bins=edges)
rate = counts / bin_width                            # spikes/s
bin_centres = edges[:-1] + bin_width / 2
</pre>

Sparse binned spikes behave like a deconvolved calcium trace: sharper in time, and noisy per
trial.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Other things to consider.</b>

<b>Timestamps.</b> Some datasets store an explicit `timestamps` array; others store a sampling
`rate` and a `starting_time`, and you reconstruct the times yourself. Everything downstream needs
real times in seconds, so check which you have &mdash; `series.timestamps` is `None` when the file
uses a rate.

<b>Lazy loading.</b> NWB data objects do not load until you index them. That is what lets you open a
50&nbsp;GB file instantly, but it means `data.std()` may fail where `np.std(data)` works. Convert
with `np.asarray()` once you know the array is small enough to hold, or slice first.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Extract neural activity. Do you have ephys units or ophys ROIs? 
Are there multiple activity types (e.g. dFF and deconvolved events) or just one? 
What do the timestamps look like? Is there more than one set of timestamps (e.g. one per plane)?

</div>

In [ ]:
# Find the neural activity in this file and extract the relevant data
# How is it formatted? What other information is available about cell activity?

In [ ]:
# Where are the timestamps located? What is the acquisition rate?

In [ ]:
# Reminder that tables become DataFrames with .to_dataframe()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Quality control: which cells or units belong in the analysis?</h3>

Segmentation and spike sorting are automated, and both over-produce. An ophys plane contains ROIs the
classifier thinks are not cell bodies; a sorted probe contains units that drift, that are barely
above noise, or that are two neurons merged. <b>The activity matrix you just loaded usually contains
all of them.</b>

Pipelines record their own verdicts. For imaging they live on the ROI table beside the masks; for
electrophysiology, on the units table. The columns differ by pipeline and by dataset &mdash; boolean
flags, continuous probabilities, morphology metrics, contamination estimates &mdash; so there is no
list to memorise. Print the columns and see what your dataset offers.

Filtering is not automatically the right move, and the criteria are yours to justify. But
<b>inheriting the unfiltered set by default is a decision you made without noticing</b>, and it is the
kind that never appears in a methods section.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Does your dataset carry per-cell or per-unit quality metrics? Report what the
columns are, how many entries each flag would exclude, and whether the activity matrix is already
filtered or contains everything.

Then decide. Whatever you choose, **state the criterion and the count you dropped** &mdash; that
sentence belongs in your methods.

</div>

In [ ]:
# EDIT: find the per-cell quality table for your dataset and print its scalar columns:
# which are flags, which are continuous scores, and what each would exclude.

In [ ]:
# EDIT: apply your QC criterion. Check the table length matches the activity
# matrix first, apply the SAME mask to every per-cell array you loaded, and print
# how many cells you dropped.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Plot one cell's trace against time, as a sanity check on what you loaded. 
Look at that trace for a few seconds before moving on. Is anything about it
surprising? Would you have noticed if you had skipped straight to the analysis?

</div>

In [ ]:
# Choose your cell deliberately rather than taking index 0 - how did you choose?
# Watch out for cells that are entirely NaN.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Find any other information needed to reproduce your classmate's plot. 
Is it showing task epochs? Stimulus presentations? 
Is there any behavioral information like reward times or running speed? 
Where do these pieces of data live in this dataset's NWB file structure?

</div>

In [ ]:
# Pull out the other pieces your chosen figure needs -- stimulus/trial tables,
# and any behavioral traces the dataset has. 
# What are the columns of the tables? Which describe *what was
# presented*, which describe *what the animal did*, and which are bookkeeping?

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Set up the main variables for the next section</h3>

Point these names at the equivalent pieces of your own NWB file. Later sections reference them,
so getting them right here saves repeating yourself &mdash; but edit anything you like as you go.
This is your notebook now.

</div>

In [ ]:
# Set up the main variables for this dataset. Later sections use these names,
# so getting them right here saves repeating yourself -- but edit anything you
# like as you go.
activity = ...              # (n_timepoints, n_cells)
timestamps = ...            # (n_timepoints,) in seconds
epochs = ...                # one row per event / trial / presentation
events = ...                # discrete timepoints when something happened

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 2: Reproduce the figure, and interrogate what it shows</h1>

You have your classmate's figure. You do not have their code, and you may not have a caption either.

<b>Before you write anything, write down what you think the figure shows.</b> One or two sentences,
in your notebook, as a claim someone could disagree with: <i>"activity is higher during X than during
Y"</i>, <i>"the response is larger on this trial type"</i>, <i>"these two signals rise together."</i>

Two reasons this comes first. It commits you to an interpretation before the data can talk you into
one &mdash; and it converts a picture into something you can actually test. A figure cannot be right
or wrong. A claim can.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Write your claim about the figure you picked, in the cell below, before you
write any code.

Be specific enough to be wrong. "There is neural activity" is not a claim; "population activity is
higher in the second half of the session" is.

</div>

_Your claim:_

<!-- write it here -->


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Now replicate it</h3>

Get the pieces the figure needs and plot them. 

As you do this, consider the following: 
* What are the decisions that go into making this figure? 
* What pieces of information are present? Are they timeseries, events, epochs? 
* Can you tell how it was filtered, if at all? 
* Are there pieces of data you would have included but they did not?
* How do the decisions made during data visualization influence your understanding of the data?


</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Plot the same data features that your classmate found in this dataset.  

What is going on in this session? 
</div>

In [ ]:
# Compute whatever your chosen figure shows, or plot the data directly.
# For a population_rate average, the mean across cells at each timepoint.
# For running speed, a single timeseries. 
# Are there discrete events? Are all available events included? 

In [ ]:
# Here is a helper function from the tutorial in case you need it 

colors = dict(zip(epochs.index, plt.cm.Pastel1.colors))

def shade_epoch_blocks(ax):
    """Shade each epoch on `ax`, one colour per epoch label.

    Epochs are the coarse structure of the session -- which stimulus block or
    task phase was running. Shading them behind a trace shows at a glance
    whether a change in activity lines up with a change in what was happening.
    """
    for label, row in epochs.iterrows():
        # zorder=0 keeps the shading BEHIND the data; alpha so the trace on top
        # stays readable. label= puts each epoch in the legend once.
        ax.axvspan(row.start_time, row.stop_time, color=colors[label],
                   alpha=0.5, zorder=0, label=label)


<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**[Optional] Exercise:** Now test the claim you wrote above.

Turn your sentence into a number you can check. If it compares epochs, compute the mean in each one,
alongside how long each epoch lasted, when in the session it happened, and what the animal was doing.
If it compares something else, compute the equivalent.

Before you look: **what would make this comparison unfair?** Write your answer down first, then see
whether the table bears it out.

Is claim as supported, contradicted, or untestable with this data? 

</div>

In [ ]:
# For each epoch compute the mean activity, and alongside it the things that
# could confound the comparison: how long the epoch lasted, how many samples
# that is, when in the session it happened, and what the animal was doing.
# Build a DataFrame with one row per epoch.

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 3: Align activity to event onsets</h1>

Let's go beyond the figure your classmate made. 

A session overview shows everything at once, which means it shows very little. To see a response
you need to **align** activity to the times when something happened, and look across repeats, not just the average.

This morning's tutorial averaged across presentations. Here we look at what the average hides.

Note that "something happened" need not be a visual stimulus. It might be a sound, an optogenetic pulse, a
reward, a lick, or the start of a trial. Anything with a repeatable onset time works the same way
&mdash; and the rest of this notebook says "event" rather than "stimulus" for that reason.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Identify events of interest.

What is happening during this session? Are there distinct stimuli, trial types, behavior events?

Explore the available conditions and select a specific type of event to examine further. 

</div>

In [ ]:
# How many times was each chosen condition presented? 
# Are there different variations on a given event type? 
# How does your selection of event types and conditions influence the amount of data you have for analysis?

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Are all of these events the same kind of event?</h3>

An event table usually contains rows that are **not equivalent trials**. Depending on the dataset
that might be first versus repeated presentations, rewarded versus unrewarded trials, different
stimulus families, trials the animal responded to versus ignored, blocks recorded before and after
a manipulation, or blank and omitted entries that are not events at all.

This matters before you align anything, for two reasons:

- **Response magnitude can differ several-fold between trial types.** Averaging them together dilutes
  the response toward whichever type is most numerous &mdash; which is often the weakest one.
- **Trial types differ in what else is happening.** Reward, licking, and arousal ride along with some
  trial types and not others, so a difference you attribute to the stimulus may not be about the
  stimulus.

Find the columns in your table that distinguish trial types, and count them.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Pick two conditions from your table and save the onset times to a variables called

| `event_times_a` | 


| `event_times_b` | 


How many repetitions are there of each condition? 

</div>

In [ ]:
# Which columns in your event table distinguish different KINDS of trial?
# Find them and count the rows of each kind. Are they balanced?

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Align the population average to
each event separately, then plot both on the same axes.

Write down your prediction first: do you expect a difference, and how large?

</div>

_Your prediction:_

<!-- write it here -->


In [ ]:
# You have a single trace (the population average) and and many event times for each condition, 
# and they do not necessarily align. We need a way to translate between them.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

To compare them we need to cut a window of data around each onset. Same
`align_to_event_times` helper as this morning's tutorial.

</div>

In [6]:
# Here is the helper function from today's tutorial, with comments added so you can follow the logic

def align_to_event_times(data, timestamps, event_times, pre=0.5, post=1.5):
    """Cut a window of data around each event time.

    data        : array with time along the first axis
    timestamps  : time of each row of data, in seconds
    event_times : times to align to, in seconds
    pre, post   : seconds before and after each event

    Returns (aligned_windows, window_time_axis) where the time axis is in
    seconds relative to the event, and there is one window per usable_cells event.
    """
    # Sampling interval. Median, not mean: one gap in the recording would
    # inflate a mean and silently shrink every window.
    dt = np.median(np.diff(timestamps))

    # Convert the requested seconds into a number of samples. int() truncates,
    # so a window that is not a whole number of samples comes out slightly
    # short -- check this if you need exact window edges.
    n_pre, n_post = int(pre / dt), int(post / dt)

    aligned_windows = []
    for event_time in event_times:
        # Index of the first sample AT OR AFTER the event. side='left' returns
        # the insertion point, so timestamps[i] >= event_time always.
        #
        # Do NOT round to the nearest sample: that pulls roughly half the
        # trials one sample EARLIER than the event, which smears the onset and
        # can make a real response look like it starts before the stimulus.
        # Landing just after is honest -- the bias is one-directional and at
        # most one sample.
        i = np.searchsorted(timestamps, event_time, side='left')

        # Skip events too close to either end of the recording to fill a whole
        # window. This drops trials SILENTLY, so compare
        # aligned_windows.shape[0] against len(event_times) afterwards.
        if i - n_pre >= 0 and i + n_post <= len(timestamps):
            # Slice is n_pre + n_post samples long. Index n_pre within the
            # window is the first sample at/after the event, i.e. t = 0.
            aligned_windows.append(data[i - n_pre:i + n_post])

    # Time axis in seconds relative to the event. Starts at -n_pre*dt, which
    # can be slightly later than -pre because of the truncation above.
    window_time_axis = np.arange(-n_pre, n_post) * dt
    return np.array(aligned_windows), window_time_axis


In [ ]:
# Feed your population average timeseries and your event times into the helper function, once for each condition
# Consider what window around each event you want to look at and adjust the input parameters to the function accordingly

In [ ]:
# What does the output look like? 
# Compare the number of windows you got back against the number of onsets you asked for. Are they the same?

In [ ]:
# Take the mean across the aligned windows for each condition and plot them on the same axis. Label them accordingly. 
# Make sure you understand the units of the x and y axes and label them as well. 

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

What are some decisions you can make in how you process and plot the data that can affect your interpretation? 

Should you subtract the pre-event baseline? Or are you interested in the baseline activity level? 

</div>

In [ ]:
# A note on baseline subtraction: exclude the sample immediately before event onset from your baseline window. 
# With binned or sampled data that sample can straddle the event,  including
# it puts part of the response into the baseline and shrinks what you
# measure. e.g. `window_time_axis < -bin_width` rather than `< 0`.


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>What about individual cells?</h3>

The population average might show one thing, but individual cells likely give a different picture. 
The activity of neurons in the brain is highly heterogeneous; understanding this diversity is a central goal of neuroscience.

But how to pick a cell to look at?

Papers often show example cells without disclosing how they were selected. 

Whatever your dataset calls them &mdash; ROIs in an imaging plane, sorted units on a probe &mdash;
taking the first one in the table is an arbitrary choice, its effectively a random sample. Selecting by how strongly
cells respond is a different choice that influences your impression of the data. Decide how you want to select cells to plot and consider the implications. 

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Choose an example cell to look at, and say how you chose it.

How does your selection criterion influence what you can learn? 

</div>

In [ ]:
# Choose an example cell - it can be random, based on activity level, or based on some metric
# Write down the rationale for your selection criterion

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Are you looking at the right signal?</b>

Most datasets ship more than one representation of the
same activity, and the choice is yours &mdash; but it is a choice, and it changes what the figures
show.

<table>
<tr><td><b>&Delta;F/F</b> (imaging)</td><td>Continuous fluorescence. Carries the indicator's rise and
decay, so a brief response is smeared forward by hundreds of milliseconds, and slow drift shared
across the field of view inflates correlations between any two cells. Every timepoint has a
value.</td></tr>
<tr><td><b>Deconvolved events</b> (imaging)</td><td>An estimate of when the cell actually fired, with
the indicator kinetics removed. Temporally tighter, and mostly exact zeros &mdash; so single-trial
estimates are much noisier even though the trial average looks cleaner.</td></tr>
<tr><td><b>Spike times</b> (electrophysiology)</td><td>Discrete times, no continuous trace at all. You
choose a bin width to get a matrix, and that width is a real analysis decision: too fine and every
bin is empty, too coarse and you lose the timing you came for.</td></tr>
</table>

None of these is necessarily right or wrong; it depends on your question. 
A question about response <i>latency</i> or duration is badly served
by &Delta;F/F; a question needing a reliable per-trial number is badly served by a sparse signal. Pick
one, say why, and if you have time run the analysis twice and compare &mdash; that comparison is
usually more informative than either result alone.

</div>

In [ ]:
# If your dataset has more than one type of signal, recreate the plot above for each. Plot them side by side. 
# What are the differences? How would it affect your interpretation?

In [ ]:
# Even if your dataset only has one signal (spike times), your choice of binning can affect the result
# Try recreating the spike times array with different bin widths and plot the average event aligned response with both versions side by side

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Raster vs PSTH</h3>

The event aligned average is also called a PSTH (peri-stimulus time histogram). 

A plot of every trial of a given condition is often called a raster. 

The raster shows every trial; the PSTH is their average. Plot them together so you can see what the
average discards.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** For one of your two conditions, plot the raster and the PSTH side by side. 

Using the same cell you've been working with, show the response for every trial as a heatmap (rows are trials, columns are timepoints). This is a raster.

Then plot the PSTH as the average across trials with the across trial variance as errorbars. 
Compute the standard error of the mean (standard deviation across trials divided by the square root of number of trials) 
and plot errorbars using matplotlib's fill_between()

</div>

In [ ]:
# Plot the raster and the PSTH side by side: every trial as a heatmap, and
# the trial average with a measure of variability acros trials.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>PSTH across cells</h3>

We have now seen average differences across conditions on average, variability across trials for one cell - how about variability across cells for one condition?

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Now compute the average response for every cell and plot the result as a heatmap, sorted by
response magnitude. How many cells respond?

</div>

In [ ]:
# Build an array of (n_cells, n_timepoints) where the value for each cell is the condition mean
# Plot the response array as a heatmap, sorted by response magnitude.


In [ ]:
# Plot it again with each cell's own pre-onset baseline subtracted, and compare the two versions. 
# How does baseline subtraction influence the interpretation? 
# Was this the appropriate thing to do given the stimulus or event conditions you are using? 

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 4: Signal and noise correlations</h1>

<h3>First, the math</h3>

The Pearson correlation between two variables $x$ and $y$ is

$$ r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}
              {\sqrt{\sum_i (x_i - \bar{x})^2}\;\sqrt{\sum_i (y_i - \bar{y})^2}} $$

In words:

1. **Center** each variable by subtracting its mean.
2. **Multiply** the centered values pointwise and sum &mdash; large and positive when they vary
   together, negative when oppositely, near zero when unrelated.
3. **Normalize** by each variable's spread, forcing the result between -1 and +1.

Compute it once by hand before running it thousands of times.

</div>

In [ ]:
# Compute the correlation between two cells' traces BY HAND, in three steps:
#   1. centre each variable (subtract its mean)
#   2. multiply the centred values pointwise and sum
#   3. normalise by the spread of each
# Then check your answer against np.corrcoef.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Two consequences that matter for everything below:

- $r$ says nothing about response **size**, only whether two things move together.
- $r$ is computed over a set of paired observations, and **how many observations you have determines
  how noisy $r$ is** &mdash; but the value itself gives you no clue how many there were.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** What does a given value of $r$ look like? Simulate pairs with known
correlations and plot them.

</div>

In [ ]:
# Simulate pairs of variables with known correlations (try 0, 0.2, 0.5, 0.9)
# and plot each as a scatter, titled with its measured r.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Two reasons neurons are correlated</h3>

- **Signal correlation.** Do they respond similarly *across conditions*? Correlate the two neurons'
  tuning curves &mdash; their average response to each condition.
- **Noise correlation.** When the *same* condition repeats, do they fluctuate together around their
  own averages? Subtract each condition's mean and correlate the residuals.

A "condition" is whatever your event table repeats: an image, a grating direction, a tone, a
photostimulation target, a task context. All that matters is that it recurs enough times to average
over.

Same data, different thing averaged over:

| | what is correlated | one observation is |
| --- | --- | --- |
| signal | condition means | one condition |
| noise | within-condition residuals | one trial |

That last column matters more than anything else in this notebook.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 1: choose which events to use</h3>

Not every event is comparable to every other. Decide which subset is a fair comparison and write down
why.

</div>

In [ ]:
# Restrict to comparable events, and write down WHY -- this choice belongs in
# your methods. Name the results:
#   `all_onset_times` -- the onset times you keep
#   `labels`     -- the chosen_condition label for each of those onset_times

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 2: one number per trial per neuron</h3>

We need a `(n_trials, n_cells)` matrix. Average each aligned window over a response window, and
subtract a **baseline** from just before onset &mdash; otherwise each trial's "response" includes
wherever the cell happened to be sitting beforehand, and those levels drift together across the
population from bleaching, arousal, and movement.

<b>Choosing the two windows is dataset-specific.</b> The response window should cover the response
your Part 3 plot showed &mdash; look at it rather than copying a number from here, since a calcium
signal and a spike rate need very different windows. The baseline window should sit in the gap
before onset, and must **exclude any stimulation artifact**: with optogenetics or electrical
stimulation the frames around the pulse can be unusable, so leave a margin on both sides.

</div>

In [ ]:
# Build the (n_trials, n_cells) response matrix `trial_response_matrix`: for each
# trial, mean activity in a response window minus a pre-onset baseline.
#
# BUILD IT IN STEPS, in separate cells, checking as you go. The solutions
# notebooks do it this way for a reason -- a shape printed at the end of a loop
# tells you almost nothing about whether the arithmetic inside was right.
#   a) pick the two windows, and print how many SAMPLES each one holds
#   b) do one trial, one cell, by hand -- print the actual values you average
#   c) one trial, all cells -- check the row length equals n_cells
#   d) loop over trials -- count how many rows came out incomplete
#   e) drop those rows from the matrix AND from the labels with the SAME mask,
#      then assert the two lengths match
#
# Note on the baseline: exclude the sample immediately before onset. With
# binned or sampled data that sample can straddle the event, so including it
# puts part of the response into the baseline and shrinks what you measure --
# `window_time_axis < -bin_width` rather than `< 0`. Then check how many
# samples are actually left: a "baseline" of one sample is not an average.


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 3: tuning curves &mdash; look before correlating</h3>

</div>

In [ ]:
# Build the condition_mean_response curves: the per-chosen_condition mean response of each cell.
# Name the chosen_condition list `conditions` and the array `condition_mean_response`,
# shaped (n_conditions, n_cells).

In [ ]:
# Plot the condition_mean_response curves before correlating anything: a few cells as lines,
# and all cells as a heatmap.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** How many numbers make up one neuron's tuning curve?

That is how many paired observations each signal correlation gets. Write it down.

</div>

In [ ]:
# How many numbers make up one condition_mean_response curve, and how many trials are available
# for the noise correlations? Print both.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 4: residuals &mdash; look before correlating</h3>

Subtract **each condition's own mean**, not the grand mean. Subtracting the grand mean would leave
the differences between conditions in the residuals, making your "noise" correlation partly a signal
correlation.

</div>

In [ ]:
# Build the residuals: subtract each chosen_condition's OWN mean from its trials.
# Name the array `residuals`, and check that its overall mean is ~0.

In [ ]:
# Plot the raw responses and the residuals for one cell, side by side, so you
# can see what subtracting the chosen_condition means removed.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 5: correlate</h3>

`np.corrcoef` correlates **rows**, so transpose to get cells rather than trials. Getting this
backwards produces a plausible matrix of entirely the wrong thing &mdash; check the output shape
against the number of cells.

</div>

In [ ]:
# Compute the two correlation matrices, `signal_corr_matrix` and `noise_corr_matrix`.
# Watch the orientation: np.corrcoef correlates ROWS.
# Take each pair once with np.triu_indices -- name the index `pairs`, and the
# extracted values `signal_values` and `noise_values`.
# Print the mean of each, with how many observations went into it.

In [ ]:
# Plot the two matrices side by side, plus signal against noise correlation
# for every pair. Scale each matrix to its own range so neither saturates.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Is this result trustworthy?</h1>

Every number so far is a point estimate with no error bar. The single most useful check: **would you
get the same answer with half the data?**

Split trials in half at random, compute the correlations on each half separately, and correlate the
two halves' answers. Split **within each condition** so both halves see every condition.

Three outcomes, and all three are informative:

- **One high, one low** &mdash; trust the high one, and say why the other is not trustworthy.
- **Both high** &mdash; you have enough data for both; proceed.
- **Both near zero** &mdash; report that. It usually means the condition variable you chose does not
  organise these neurons' responses, however well-balanced it looked in the inventory. That is a
  real result about your dataset, and it is a better README than a matrix you cannot defend.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Before running it &mdash; which do you expect to be more reliable, signal or
noise correlations? Look back at the observation counts you wrote down.

</div>

In [ ]:
def signal_and_noise_correlations(responses, labels):
    """Signal and noise correlation matrices from a set of trials.

    responses : (n_trials, n_cells) one response value per trial per cell
    labels    : (n_trials,) which condition each trial belongs to

    Signal correlation = do two cells prefer the same conditions?
    Noise correlation  = do two cells co-vary trial to trial WITHIN a
                         condition, once the condition mean is removed?
    """
    conditions = np.unique(labels)

    # TUNING: one row per condition, holding that condition's mean response for
    # every cell. Averaging over trials is what removes trial-to-trial noise
    # and leaves the stimulus preference -- the "signal".
    condition_means = np.vstack([responses[labels == c].mean(axis=0)
                                 for c in conditions])

    # RESIDUALS: each trial minus its own condition's mean. What remains is
    # everything the condition does NOT explain -- the "noise". Subtracting the
    # condition mean is essential: skip it and the condition structure leaks
    # into the noise matrix and inflates it.
    residuals = responses.astype(float).copy()
    for c in conditions:
        in_condition = labels == c
        residuals[in_condition] -= responses[in_condition].mean(axis=0)

    # .T because np.corrcoef correlates ROWS: we want cell-by-cell matrices,
    # and cells are the columns of both arrays.
    #
    # Note the very different sample sizes feeding these two matrices: signal
    # is estimated from len(conditions) numbers per cell, noise from
    # len(labels) trials. That asymmetry is why they differ so much in
    # reliability even though both render as equally convincing heatmaps.
    return np.corrcoef(condition_means.T), np.corrcoef(residuals.T)


In [ ]:
# Split-half split_half_reliability. Split the trials in half WITHIN each chosen_condition,
# compute the correlation matrices on each half with `signal_and_noise_correlations`, and
# correlate the two halves' answers (scipy.stats.spearmanr on `pairs`).
# Repeat ~10 times; report the mean and spread for signal and for noise.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Does the signal you chose change the answer?</h3>

Everything so far used one representation of activity. If your dataset provides a second one, repeat
the whole chain on it and compare the numbers that matter. If it provides only one, note that and
move on.

To repeat the chain you need the response-matrix construction as a reusable function rather than a
one-off block &mdash; so wrap it, the same way you wrapped the correlations.

</div>

In [ ]:
def trial_by_cell_responses(A):
    """Build the (n_trials, n_cells) baseline-subtracted response matrix.

    One number per trial per cell: mean activity in the response window minus
    mean activity in the baseline window. Every correlation below is computed
    from this matrix, so both window choices propagate into every later result.
    """
    response_rows = []
    for t0 in all_onset_times:
        # Boolean masks selecting the samples in each window for this trial.
        # >= start and < end so the two windows never share a sample.
        in_response = (timestamps >= t0 + response_window[0]) & (timestamps < t0 + response_window[1])
        in_baseline = (timestamps >= t0 + baseline_window[0]) & (timestamps < t0 + baseline_window[1])

        # nanmean, not mean: a single all-NaN cell would otherwise propagate
        # NaN across the whole row and silently cost you every trial.
        # A trial at the very start of the recording can have an empty
        # baseline window -- fill it with NaN and drop it below.
        response_rows.append(np.nanmean(A[in_response], axis=0) - np.nanmean(A[in_baseline], axis=0)
                    if in_response.sum() and in_baseline.sum()
                    else np.full(A.shape[1], np.nan))

    responses = np.array(response_rows)

    # Drop trials with any missing cell. Report the count if it is not zero:
    # trials vanishing here is exactly the kind of silent loss to check for.
    return responses[~np.isnan(responses).any(axis=1)]


In [ ]:
# Run the whole chain on each activity representation your dataset has, and
# compare: mean signal and noise correlation, their split-half reliabilities,
# and what fraction of the single-trial responses are exactly zero.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>One column may not be the whole condition.</b>

A column can look like a clean condition variable &mdash; many levels, perfectly balanced &mdash;
while the stimulus varied in some <i>other</i> way at the same time. Two trials sharing that column's
value are then not repeats of the same thing, and averaging them together destroys the tuning you
were trying to measure.

Receptive-field mapping is the classic case: orientation is balanced, but the stimulus also moves
around the screen, so "144 repeats of 45&deg;" is really a handful of repeats at each of many
positions. The same trap appears whenever a design crosses two factors and you only notice one.

Check for it by asking what else varies across the trials you just called identical. Group by your
condition column, look at the other columns within a group, and see whether they are constant. If
they are not, either restrict to one level of the other factor, or make the condition the
<i>combination</i> of both.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Signal correlations need a condition that repeats. Does your dataset have
one?

Inventory the candidate columns: how many distinct values, how many repeats, how balanced.

Then answer **two separate questions**, because they can disagree:

1. **Is the analysis possible?** Does some column have enough conditions with enough repeats?
2. **Is it meaningful?** Does that column label something you would expect neurons to be tuned
   *to*, in a way that a correlation across condition means would capture?

A column can pass the first test and fail the second. State a verdict on both, and check it against
your reliability numbers.

</div>

In [ ]:
# Inventory the candidate chosen_condition columns in your event table: for each,
# how many distinct values, the repeats of the least and most common, and how
# balanced. Then state your verdict on both questions above.

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Summary</h1>

<h3>The process</h3>

1. **Find out what is in the file** before analyzing it &mdash; and check that the dataset supports
   your question. Sometimes the answer is no.
2. **Plot the data after each transformation.** Single trials before averages; tuning curves before
   correlations.
3. **Name every decision.** Event subset, condition column, response window, baseline. Each is a
   fork, and each belongs in your methods.
4. **Try to break your own result.** Split the data in half and see if the answer survives.
5. **Let the dataset answer back.** If the check says your result is noise, or the dataset has no
   variable that supports your question, that is the finding. Report it rather than reaching for the
   analysis you planned to run.

<h3>Traps this notebook demonstrated</h3>

| trap | how you catch it |
| --- | --- |
| A result from few observations looks like one from many | split-half reliability |
| A well-balanced condition variable that means nothing | reliability, not the inventory |
| A condition column that hides a second varying factor | group by it, check what else moves |
| Analyzing units that should have been dropped | select on quality columns, and say so |
| A helper function silently drops data | compare output shape to input |
| A column exists but carries no information | check that it actually varies |
| One bad trial turns every cell's score into NaN | count your NaNs; use `nanmean` |
| Epoch comparisons confounded with time and behavior | check durations, order, behavior |
| An example cell chosen to look good | state your selection rule |
| Data looks absent but is stored elsewhere | look in every container first |
| An index from an earlier cell after reshaping the data | re-derive indices, never carry them |

<h3>Why this matters</h3>

You can generate an analysis faster than you can validate one. The only defense is to know your data
well enough that a wrong answer looks wrong to **you** &mdash; because it will not look wrong to the
code, and it will not look wrong on the plot.

</div>